[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C09_Reasoning_TTC_Course/06_efficient_reasoning/06_overthinking_budget.ipynb)

# 06 · Overthinking 与预算控制：长思维链的效率经济学

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span> 纯 numpy/matplotlib 模拟，自包含，所有 cell 秒级跑完。

**本 notebook 你将完成：**

1. 构造一个**思考轨迹模拟器**：两态 Markov 链（修复率/破坏率随难度变化）+ 不稳定的错误答案；
2. 度量 **overthinking**：第一个正确答案出现位置的分布、"再想下去被改错"的概率、按难度分层的 $\mathrm{acc}(t)$ 边际收益曲线；
3. 在**同一 token 预算**下对比三种停止策略：固定长度 / answer-stability 早停 / 难度路由；
4. 模拟 **budget forcing**：强行截断 vs 追加 "Wait" 延长，观察对各难度层的差异化影响；
5. 画 **accuracy-per-token 帕累托前沿**并提取非支配点；
6. 4 道 ✏️ 练习巩固核心度量与策略。

参考：[Chen 2024] *Do NOT Think That Much for 2+3=?* (arXiv:2412.21187)、[Muennighoff 2025] *s1: Simple test-time scaling* (arXiv:2501.19393)、[Kimi k1.5] (arXiv:2501.12599)、[Snell 2024] (arXiv:2408.03314)。

## 1 · 思考轨迹模拟器：把"想下去"建成两态 Markov 链

把一条长思维链抽象成**中间答案序列**：模型每多想一步（≈ 一段推理 + 一个当前答案，折算 `TOKENS_PER_STEP` 个 token），当前答案的对/错按两态 Markov 链演化：

- **修复率 `fix`**（错 → 对）：难题低、简单题高 —— 多想一步把错答案改对的概率；
- **破坏率 `brk`**（对 → 错）：所有难度都非零 —— "想歪改错"（self-doubt 把对的改掉）；
- **首步正确率 `p0`**：简单题第一个答案往往已正确（[Chen 2024] 的核心观察）。

讲解页第 4 节推过：$\mathrm{acc}(t) = \pi + (\mathrm{acc}(0)-\pi)(1-f-b)^t$，平稳值 $\pi = f/(f+b)$。参数设计上让 **easy 的 $p_0 > \pi$**（继续想净亏）、**hard 的 $p_0 \ll \pi$**（长思考真有用）。另外让错误答案**不稳定**（高概率换来换去），正确答案天然稳定 —— 这是后面 answer-stability 早停能工作的物理基础。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

TOKENS_PER_STEP = 48             # 每个思考步折算的 token 数
T_MAX = 64                       # 最长思考 64 步 ≈ 3k token
DIFFS = ["easy", "medium", "hard"]
PARAMS = {                       # p0: 首步即正确 | fix: 错→对(修复率) | brk: 对→错(破坏率, 想歪改错)
    "easy":   dict(p0=0.90, fix=0.30, brk=0.060),   # 平稳值 π=0.83 < p0 → 越想越差
    "medium": dict(p0=0.35, fix=0.15, brk=0.030),   # π=0.83 > p0 → 上升后饱和
    "hard":   dict(p0=0.05, fix=0.06, brk=0.015),   # π=0.80 ≫ p0，混合慢 → 需要长思考
}

def simulate_trace(diff, T=T_MAX, rng=rng):
    """模拟一条思考轨迹：返回长度 T 的中间答案序列（正确答案恒为 0，错误答案为 1..9）。"""
    p = PARAMS[diff]
    ans = np.empty(T, dtype=np.int64)
    correct = rng.random() < p["p0"]
    wrong = int(rng.integers(1, 10))
    for t in range(T):
        ans[t] = 0 if correct else wrong
        if correct:
            if rng.random() < p["brk"]:            # 想歪：把对的改错，换一个错误答案
                correct, wrong = False, int(rng.integers(1, 10))
        else:
            if rng.random() < p["fix"]:            # 修复：把错的改对
                correct = True
            elif rng.random() < 0.6:               # 错误答案不稳定：大概率换一个错的
                wrong = int(rng.integers(1, 10))
    return ans

N_PER = 250
difficulty = np.repeat(DIFFS, N_PER)                          # (750,)
traces = np.stack([simulate_trace(d) for d in difficulty])    # (750, 64) 中间答案
is_correct = (traces == 0)                                    # (750, 64) 步 t 当前答案是否正确

print("traces:", traces.shape, "| 满长思考 ≈", T_MAX * TOKENS_PER_STEP, "token/题")
for d in DIFFS:
    m = difficulty == d
    print(f"{d:>6}: 首步答对率 {is_correct[m, 0].mean():.2f} → 末步答对率 {is_correct[m, -1].mean():.2f}")

## 2 · 度量 overthinking：三张诊断图

[Chen 2024] 的两个标志性发现，翻译成可计算的度量：

1. **first correct position**：第一个正确答案出现的步索引分布 —— easy 题应集中在 0（第一个答案已正确，后面全是 overthinking）；
2. **acc(t) 边际收益曲线**：若在步 $t$ 强制停止、提取当前答案的准确率 —— 按难度分层，复现"简单题早停无损甚至更好、难题需要长思考"；
3. **改错风险**：步 $t$ 已答对的题里，最终答案被改错的比例 —— 继续思考的纯下行风险。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
xs_tok = np.arange(T_MAX) * TOKENS_PER_STEP

# (a) 第一个正确答案出现位置的分布
for d in DIFFS:
    m = difficulty == d
    ever = is_correct[m].any(axis=1)
    first = np.argmax(is_correct[m], axis=1)[ever]
    axes[0].hist(first, bins=np.arange(0, T_MAX + 2) - 0.5, alpha=0.55,
                 label=f"{d}（从未答对 {1 - ever.mean():.0%}）")
axes[0].set(title="first correct position", xlabel="思考步 t", ylabel="题数")

# (b) acc(t)：若在步 t 强制停止
for d in DIFFS:
    axes[1].plot(xs_tok, is_correct[difficulty == d].mean(axis=0), label=d)
axes[1].set(title="acc(t)：边际收益按难度分层", xlabel="思考 token", ylabel="此刻停止的准确率")

# (c) 改错风险：P(最终错 | 步 t 已对)
for d in DIFFS:
    c = is_correct[difficulty == d]
    risk = [(c[:, t] & ~c[:, -1]).sum() / max(c[:, t].sum(), 1) for t in range(T_MAX)]
    axes[2].plot(xs_tok, risk, label=d)
axes[2].set(title="P(最终被改错 | 步 t 已正确)", xlabel="思考 token", ylabel="改错概率")

for ax in axes: ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

acc_e = is_correct[difficulty == "easy"]
print(f"easy: 首步 acc {acc_e[:, 0].mean():.3f} → 末步 {acc_e[:, -1].mean():.3f}"
      f"（继续想净变化 {acc_e[:, -1].mean() - acc_e[:, 0].mean():+.3f} ← overthinking 的代价）")
acc_h = is_correct[difficulty == "hard"]
print(f"hard: 首步 acc {acc_h[:, 0].mean():.3f} → 末步 {acc_h[:, -1].mean():.3f}（长思考的收益）")

## 3 · 三种停止策略：同预算下谁更准？

固定平均预算（每题 24 步 ≈ 1152 token），对比：

| 策略 | 机制 | 对应现实技术 |
|---|---|---|
| `fixed-24` | 所有题一律思考 24 步 | 统一 `max_thinking_tokens` |
| `stability-k6` | 当前答案连续 6 步不变即停 | answer consistency / early exit |
| `routing` | 便宜的难度估计器（80% 准，错则混淆到相邻档）→ easy 6 步 / medium 24 步 / hard 42 步 | difficulty routing（[Snell 2024] compute-optimal 的离散版） |

routing 的三档预算在三档等量时均值恰为 24 步 —— 这是**同预算公平比较**的关键设定。

In [ ]:
def stop_fixed(trace, L):
    """固定长度：思考 L 步后停止。返回 (答案, 实际步数)。"""
    L = min(L, len(trace))
    return trace[L - 1], L

def stop_stability(trace, k, cap=T_MAX):
    """answer-stability 早停：当前答案连续 k 步不变即停；到 cap 步强制停。"""
    run = 1
    for t in range(1, min(cap, len(trace))):
        run = run + 1 if trace[t] == trace[t - 1] else 1
        if run >= k:
            return trace[t], t + 1
    t = min(cap, len(trace)) - 1
    return trace[t], t + 1

# 便宜的难度估计器：80% 正确，出错只会混淆到相邻档（模拟小模型预判/置信度信号）
est = difficulty.copy()
flip = rng.random(len(est)) < 0.20
adj = {"easy": ["medium"], "medium": ["easy", "hard"], "hard": ["medium"]}
est[flip] = [rng.choice(adj[d]) for d in est[flip]]
print(f"难度估计器准确率: {(est == difficulty).mean():.2f}")

BUDGET = 24
ALLOC = {"easy": 6, "medium": 24, "hard": 42}        # (6+24+42)/3 = 24 步

def stop_routing(trace, est_diff, alloc=ALLOC):
    return stop_fixed(trace, alloc[est_diff])

policies = [("fixed-24",     lambda tr, i: stop_fixed(tr, BUDGET)),
            ("stability-k6", lambda tr, i: stop_stability(tr, 6)),
            ("routing",      lambda tr, i: stop_routing(tr, est[i]))]
for name, fn in policies:
    outs = [fn(traces[i], i) for i in range(len(traces))]
    acc = np.mean([a == 0 for a, _ in outs])
    steps = np.mean([s for _, s in outs])
    per = {d: np.mean([outs[i][0] == 0 for i in range(len(outs)) if difficulty[i] == d]) for d in DIFFS}
    print(f"{name:>13}: acc={acc:.3f} | 平均 {steps:5.1f} 步 ≈ {steps * TOKENS_PER_STEP:5.0f} token | "
          + "  ".join(f"{d} {per[d]:.2f}" for d in DIFFS))

## 4 · Budget forcing：截断 vs 追加 "Wait"

[Muennighoff 2025] 的 s1 在解码期双向控制思考长度：**截断**（到预算即强插 "Final Answer:"）与**延长**（抑制结束符、追加 `"Wait"`，模型继续自查）。我们模拟：

- 每题有一个**自然停止长度** `nat_len`（模型自己决定何时结束思考；与难度相关，但 hard 往往停得太早 —— 这正是 "Wait" 有用的前提）；
- **截断**：实际思考 = min(自然长度, cap)；
- **延长**：每追加一次 "Wait" 多想 8 步，实际思考 = min(自然长度 + 8n, T_MAX)。

预期复现 s1 的图景：延长对 hard 单调有益（test-time scaling 干预曲线），对 easy 无收益甚至为负；截断对 easy 几乎无损。

In [ ]:
NAT = {"easy": (10, 3), "medium": (22, 5), "hard": (26, 6)}    # 自然停止长度 (均值, 标准差)
nat_len = np.clip(np.array([rng.normal(*NAT[d]) for d in difficulty]).round().astype(int), 2, T_MAX)

def acc_if_stop(stop_steps):
    """在每题指定步数处停止时，各题是否答对（bool 数组）。"""
    idx = np.clip(stop_steps, 1, T_MAX) - 1
    return traces[np.arange(len(traces)), idx] == 0

base = acc_if_stop(nat_len)
print("自然停止 baseline:")
for d in DIFFS:
    m = difficulty == d
    print(f"  {d:>6}: acc={base[m].mean():.3f} | 平均思考 {nat_len[m].mean():4.1f} 步")

caps, waits = [4, 8, 16, 24, 32], [0, 1, 2, 4, 6]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for d in DIFFS:
    m = difficulty == d
    axes[0].plot(caps, [acc_if_stop(np.minimum(nat_len, c))[m].mean() for c in caps], marker="o", label=d)
    axes[1].plot([w * 8 for w in waits],
                 [acc_if_stop(np.minimum(nat_len + w * 8, T_MAX))[m].mean() for w in waits], marker="o", label=d)
axes[0].set(title="截断：think ≤ cap", xlabel="cap（步）", ylabel="accuracy")
axes[1].set(title='延长：追加 "Wait" ×n（每次 +8 步）', xlabel="额外思考步数", ylabel="accuracy")
for ax in axes: ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5 · accuracy-per-token 帕累托前沿

把每个 (策略, 参数) 组合评测成一个点 $(\bar n_{\text{token}}, \mathrm{acc})$，扫参数得到点云，再提取**非支配点**（不存在另一点 token 更少且 acc 更高）。这是效率感知评测的标准呈现：单一比值 acc/token 隐含线性可换假设，帕累托图不做该假设。

预期：**stability 系策略支配几乎所有 fixed 与 routing 点** —— 它"边想边看答案是否收敛"，等价于一个近似 oracle 的难度信号，自适应地少花 token 还多拿 acc。注意这是模拟器的理想化之处：现实中从思考流提取中间答案本身有噪声与成本（答案解析失败、提取器调用开销），stability 不会这么接近 oracle，但"自适应 > 统一预算"的排序在真实系统中反复成立。

In [ ]:
def evaluate_policy(stop_fn):
    outs = [stop_fn(traces[i], i) for i in range(len(traces))]
    acc = float(np.mean([a == 0 for a, _ in outs]))
    tok = float(np.mean([s for _, s in outs]) * TOKENS_PER_STEP)
    return tok, acc

points = []                                            # (平均 token, acc, 标签)
for L in [2, 4, 8, 16, 24, 32, 48, 64]:
    tok, acc = evaluate_policy(lambda tr, i, L=L: stop_fixed(tr, L));        points.append((tok, acc, f"fixed-{L}"))
for k in [2, 3, 4, 6, 8, 10]:
    tok, acc = evaluate_policy(lambda tr, i, k=k: stop_stability(tr, k));    points.append((tok, acc, f"stab-{k}"))
for s in [0.4, 0.7, 1.0, 1.5]:
    alloc = {d: max(2, min(T_MAX, int(ALLOC[d] * s))) for d in DIFFS}
    tok, acc = evaluate_policy(lambda tr, i, a=alloc: stop_routing(tr, est[i], a)); points.append((tok, acc, f"route-x{s}"))

def is_dominated(p, pts):
    return any(q[0] <= p[0] and q[1] >= p[1] and (q[0] < p[0] or q[1] > p[1]) for q in pts)
frontier = sorted(p for p in points if not is_dominated(p, points))

plt.figure(figsize=(7, 4.5))
colors = {"fixed": "tab:gray", "stab": "tab:blue", "route": "tab:orange"}
seen = set()
for tok, acc, lab in points:
    fam = lab.split("-")[0]
    plt.scatter(tok, acc, c=colors[fam], s=32, label=fam if fam not in seen else None); seen.add(fam)
plt.plot([p[0] for p in frontier], [p[1] for p in frontier], "k--", lw=1, label="Pareto frontier")
plt.xlabel("平均思考 token / 题"); plt.ylabel("accuracy"); plt.title("accuracy-per-token 帕累托前沿")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print("前沿上的策略:")
for tok, acc, lab in frontier:
    print(f"  {lab:>10}: {tok:5.0f} token → acc {acc:.3f}")

---
## ✏️ 练习 1：实现 `first_correct_position`

输入一条思考轨迹 `trace`（一维 int 数组，正确答案为 0），返回**第一个正确答案出现的步索引**；若整条轨迹从未正确，返回 `-1`。这是 [Chen 2024] outcome efficiency 的分子来源。

**提示**：`np.nonzero(trace == 0)[0]` 给出所有正确位置，空数组说明从未正确。5 行以内。

In [ ]:
def first_correct_position(trace):
    # TODO: 返回 trace 中第一个 == 0 的索引（int）；若不存在返回 -1
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert first_correct_position(np.array([3, 0, 0, 5])) == 1
assert first_correct_position(np.array([0, 0])) == 0          # 首步即对
assert first_correct_position(np.array([7, 7, 7])) == -1      # 从未答对
med = {}
for d in DIFFS:
    pos = [first_correct_position(tr) for tr, dd in zip(traces, difficulty) if dd == d]
    med[d] = float(np.median([p for p in pos if p >= 0]))
assert med["easy"] < med["medium"] < med["hard"], med         # 首达位置随难度后移
print("✅ 练习 1 通过 | 中位首达步:", med)

## ✏️ 练习 2：实现 `stability_stop_ex`

不翻上文，自己实现 answer-stability 早停：`stability_stop_ex(trace, k)` 从头扫描，一旦当前答案**连续 k 步相同**就立即停止，返回 `(answer, stop_step)`，其中 `stop_step` = 停止位置索引 + 1（实际思考步数）；若扫到结尾仍未稳定，返回 `(最后一步答案, 全长)`。

**提示**：维护 run-length 计数器：与上一步答案相同则 +1，否则重置为 1；首次到 k 即停。注意 `k=1` 应在第 1 步就停；与 `None` 比较 numpy 标量有坑，建议首步直接置 `run=1`。约 10 行。

In [ ]:
def stability_stop_ex(trace, k):
    # TODO: 连续 k 步答案相同即停，返回 (int(answer), stop_step)
    # 未稳定则返回 (int(trace[-1]), len(trace))
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert stability_stop_ex(np.array([5, 5, 5, 5, 5]), 3) == (5, 3)      # 平稳轨迹：第 3 步即早停
assert stability_stop_ex(np.array([1, 1, 2, 2, 2, 2]), 3) == (2, 5)   # run 被打断后重新累计
assert stability_stop_ex(np.array([1, 2, 1, 2, 1, 2]), 3) == (2, 6)   # 波动轨迹：不提前停，跑满全长
assert stability_stop_ex(np.array([9, 4]), 1) == (9, 1)               # k=1：第 1 步就停
print("✅ 练习 2 通过")

## ✏️ 练习 3：实现 `budgeted_eval` —— 同预算公平对比

实现 `budgeted_eval(traces, difficulty_est, budget, policy)`：在**平均每题 `budget` 步**的约束下评测停止策略，返回 `(accuracy, avg_steps)`（均为 float）。

- `policy="fixed"`：每题一律思考 `budget` 步；
- `policy="routing"`：按估计难度分配 `easy: budget-18 / medium: budget / hard: budget+18`（下限 clip 到 2、上限 `T_MAX`）—— 三档等量时均值恰为 `budget`。

**提示**：复用上文的 `stop_fixed`；约 12 行。同预算下 routing 应 ≥ fixed —— 把简单题省下的 token 转移给难题（讲解页第 6 节的背包直觉）。

In [ ]:
def budgeted_eval(traces, difficulty_est, budget, policy):
    # TODO: policy == "fixed"   -> 每题 stop_fixed(trace, budget)
    #       policy == "routing" -> 按 difficulty_est[i] 查 {easy: budget-18, medium: budget, hard: budget+18}
    #                              （clip 到 [2, T_MAX]）再 stop_fixed
    # 返回 (accuracy, avg_steps)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
acc_f, st_f = budgeted_eval(traces, est, 24, "fixed")
acc_r, st_r = budgeted_eval(traces, est, 24, "routing")
assert abs(st_f - 24) < 1e-9                  # fixed 恰好用满预算
assert st_r <= 24 + 1.5                       # routing 平均开销不超预算（估计噪声留小余量）
assert acc_r >= acc_f                         # 同预算下：路由 ≥ 均匀固定长度
acc_f64, _ = budgeted_eval(traces, est, 64, "fixed")
assert acc_f64 <= acc_r + 0.05                # 全预算拉满也不应显著超过路由（hard 已饱和、easy 在变差）
print(f"✅ 练习 3 通过 | fixed acc={acc_f:.3f}  routing acc={acc_r:.3f} ({acc_r - acc_f:+.3f})")

## ✏️ 练习 4：`acc_per_kilotoken` 与帕累托前沿提取

实现两个效率评测工具：

1. `acc_per_kilotoken(acc, avg_tokens)`：每千 token 准确率 = $\mathrm{acc} / (\bar n / 1000)$；
2. `pareto_frontier(points)`：输入 `[(avg_tokens, acc), ...]`，返回**非支配点**按 token 升序排序的列表。点 $p$ 被 $q$ 支配 ⟺ $q$ 的 token ≤ $p$ 且 acc ≥ $p$、且至少一项严格（即 $q \ne p$）。

**提示**：这是非支配排序的简化版（只取第一层前沿）；N 小，$O(N^2)$ 两层循环足矣。注意**完全重复的点互不支配，应同时保留**——"至少一项严格"等价于"作为元组 `q != p`"。约 10 行。

In [ ]:
def acc_per_kilotoken(acc, avg_tokens):
    # TODO: 每 1000 token 的准确率
    raise NotImplementedError

def pareto_frontier(points):
    # TODO: 返回非支配点列表，按 (token, acc) 升序排序
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
assert abs(acc_per_kilotoken(0.8, 400) - 2.0) < 1e-9
pts = [(100, 0.50), (200, 0.60), (150, 0.40), (200, 0.55), (300, 0.60)]
assert pareto_frontier(pts) == [(100, 0.50), (200, 0.60)]
assert pareto_frontier([(50, 0.3), (50, 0.3)]) == [(50, 0.3), (50, 0.3)]   # 重复点互不支配
front = pareto_frontier([(t, a) for t, a, _ in points])                    # 第 5 节的点云
assert all(front[i][0] <= front[i+1][0] and front[i][1] <= front[i+1][1] for i in range(len(front) - 1))
assert all(not is_dominated((t, a, ""), points) for t, a in front)         # 与第 5 节实现一致
print("✅ 练习 4 通过 | 前沿点数:", len(front))

---
## 📖 参考答案

In [ ]:
# 练习 1 参考答案（先自己做，再对照）
def first_correct_position(trace):
    hits = np.nonzero(trace == 0)[0]
    return int(hits[0]) if len(hits) else -1

In [ ]:
# 练习 2 参考答案（先自己做，再对照）
def stability_stop_ex(trace, k):
    run = 1
    if run >= k:                                  # k=1：第 1 步就停
        return int(trace[0]), 1
    for t in range(1, len(trace)):
        run = run + 1 if trace[t] == trace[t - 1] else 1
        if run >= k:
            return int(trace[t]), t + 1
    return int(trace[-1]), len(trace)

In [ ]:
# 练习 3 参考答案（先自己做，再对照）
def budgeted_eval(traces, difficulty_est, budget, policy):
    alloc = {"easy": max(2, budget - 18), "medium": budget, "hard": min(T_MAX, budget + 18)}
    accs, steps = [], []
    for i in range(len(traces)):
        L = budget if policy == "fixed" else alloc[difficulty_est[i]]
        a, s = stop_fixed(traces[i], L)
        accs.append(a == 0)
        steps.append(s)
    return float(np.mean(accs)), float(np.mean(steps))

In [ ]:
# 练习 4 参考答案（先自己做，再对照）
def acc_per_kilotoken(acc, avg_tokens):
    return acc / (avg_tokens / 1000)

def pareto_frontier(points):
    def dominated(p):
        return any(q[0] <= p[0] and q[1] >= p[1] and q != p for q in points)
    return sorted(p for p in points if not dominated(p))

---
## 🎯 真实数据胶囊题：真实步数上的预算强制(budget forcing)

更长推理不总更好：超过题目需要的步数是“过度思考”，浪费 token。用真实 GSM8K 步数，实现按 token 预算截断的准确率曲线，找到覆盖大多数题的高性价比预算。

> 本模块新增的**真实数据**练习：自包含、用真实 GSM8K 把本章方法跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.reasoning_ttc_data"); os.makedirs(CACHE,exist_ok=True)
def _f(url,fn):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    return p
def gsm8k(n=300):
    p=_f("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]
def gold(a): return a.split("####")[-1].strip().replace(",","")
def steps(a): return max(1, a.count("<<"))   # 真实推理步数代理

rows=gsm8k(300); need=np.array([steps(r["answer"]) for r in rows])
print(f"真实题目所需步数: 中位={np.median(need):.0f}, 90分位={np.percentile(need,90):.0f}")

**练习**：实现 `coverage_at_budget(need, budget)`：返回“所需步数 <= budget”的题目比例(即给定步数预算能正确完成的覆盖率)。

In [ ]:
def coverage_at_budget(need, budget):
    # TODO: mean(need <= budget)
    raise NotImplementedError


In [ ]:
# 自测
c_med=coverage_at_budget(need, np.median(need))
c_p90=coverage_at_budget(need, np.percentile(need,90))
assert c_p90 > c_med, "更大预算覆盖更多题"
assert c_p90 >= 0.9, "90分位预算应覆盖~90%题"
# 找覆盖95%的最小预算
b95=min(b for b in range(1, int(need.max())+1) if coverage_at_budget(need,b)>=0.95)
print(f"覆盖95%题目的步数预算={b95} (超此预算多为过度思考) ✓")


### 📖 参考答案

In [ ]:
def coverage_at_budget(need, budget):
    return float(np.mean(np.asarray(need) <= budget))
print("✓ budget forcing：给够大多数题的预算，砍掉长尾过度思考")